# Demo 3: Dagster – Software-Defined Assets

**Kapcsolódó diák:** 28–29 (Dagster architektúra, @asset, Freshness Policy)

Ez a notebook a következő témákat fedi le:

1. Task-centric vs asset-centric szemléletmód
2. Az első `@asset` – legegyszerűbb példa
3. Asset függőségek – upstream/downstream lánc
4. Resource fogalma – konfigurálható kapcsolatok
5. `materialize()` – az asset-ek előállítása
6. Metadata output – megfigyelhetőség
7. `FreshnessPolicy` – mikor "elavult" egy asset?
8. Sensor – event-driven materializáció
9. `Definitions` – a projekt összefoglaló konfigurációja
10. Összefoglalás – mikor válasszuk a Dagster-t?

In [1]:
!pip install dagster dagster-duckdb duckdb

  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached grpcio-1.80.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.8 kB)
Using cached grpcio-1.80.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (6.8 MB)
Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.63.0
    Uninstalling grpcio-1.63.0:
      Successfully uninstalled grpcio-1.63.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.24.0 requires protobuf<5.0,>=3.19, but you have protobuf 6.33.6 which is incompatible.
googleapis-common-protos 1.63.0 requi

## 1. Task-centric vs asset-centric szemléletmód

**Analógia:**

| Eszköz | Szemlélet | Analógia |
|--------|-----------|----------|
| Airflow / Prefect | task-centric | **gyártósor** – az utasításokat hajtja végre |
| Dagster | asset-centric | **raktárrendszer** – az adatobjektumokat tartja nyilván |

A task-centric rendszerben azt definiálod, **mit csináljon** a pipeline.
Az asset-centric rendszerben azt definiálod, **mit hozzon létre** – a futtatási sorrend ebből következik.

In [2]:
# Task-centric szemlélet (Airflow/Prefect stílusban) – csak string demo, nincs import
task_centric_pseudocode = '''
# Airflow / Prefect stílus: UTASÍTÁSOK sorozata
@task
def fetch_data():      ...  # 1. lépés: adatot tölt
@task
def clean_data():      ...  # 2. lépés: tisztít
@task
def aggregate_data():  ...  # 3. lépés: aggregál

# A sorrendet NEKED kell megadni:
fetch_data() >> clean_data() >> aggregate_data()
'''

# Asset-centric szemlélet (Dagster stílus) – csak string demo
asset_centric_pseudocode = '''
# Dagster stílus: ADATOBJEKTUMOK definíciója
@asset
def raw_orders():        ...  # ez az objektum: nyers rendelések

@asset
def validated_orders(raw_orders):  # upstream: raw_orders
    ...                              # sorrend AUTOMATIKUS a névből

@asset
def order_summary(validated_orders):  # upstream: validated_orders
    ...
'''

print('--- Task-centric (Airflow stílus) ---')
print(task_centric_pseudocode)
print('--- Asset-centric (Dagster stílus) ---')
print(asset_centric_pseudocode)

--- Task-centric (Airflow stílus) ---

# Airflow / Prefect stílus: UTASÍTÁSOK sorozata
@task
def fetch_data():      ...  # 1. lépés: adatot tölt
@task
def clean_data():      ...  # 2. lépés: tisztít
@task
def aggregate_data():  ...  # 3. lépés: aggregál

# A sorrendet NEKED kell megadni:
fetch_data() >> clean_data() >> aggregate_data()

--- Asset-centric (Dagster stílus) ---

# Dagster stílus: ADATOBJEKTUMOK definíciója
@asset
def raw_orders():        ...  # ez az objektum: nyers rendelések

@asset
def validated_orders(raw_orders):  # upstream: raw_orders
    ...                              # sorrend AUTOMATIKUS a névből

@asset
def order_summary(validated_orders):  # upstream: validated_orders
    ...



### Mi a különbség?

- **Task-centric** (`fetch_data >> clean_data`): a DAG-ot **te írod meg** – "mit csináljon".
- **Asset-centric** (`def validated_orders(raw_orders)`): a DAG-ot **Dagster vezeti le** a paraméternevekből – "mit hozzon létre".

Ez azt jelenti, hogy Dagsterben sosem téveszthetsz el egy dependency-t:
ha `validated_orders` kell `order_summary`-hoz, azt a Python szignatúra garantálja.

## 2. Az első `@asset` – legegyszerűbb példa

A `@asset` dekorátor egy egyszerű Python függvényre kerül.
A függvény neve lesz az asset neve – ez azonosítja az adatobjektumot a rendszerben.
A visszatérési értéket Dagster elmenti (materializálja) és nyilvántartja.

In [3]:
# Bare minimum @asset: raw_orders asset, visszaad egy listát
try:
    from dagster import asset

    @asset
    def raw_orders():                          # függvénynév = asset neve
        """Nyers rendelési adatok (szimulált)."""
        # Szimuláljuk a forrásrendszerből érkező adatot
        orders = [
            {'order_id': 1, 'amount': 150.0, 'status': 'completed'},
            {'order_id': 2, 'amount': 320.5, 'status': 'pending'},
            {'order_id': 3, 'amount':  -10.0, 'status': 'cancelled'},  # hibás sor
        ]
        return orders                              # ez lesz az asset értéke

    print(f'Asset neve: {raw_orders.key}')
    print(f'Leírás:     {raw_orders.description}')
    print('raw_orders asset sikeresen definiálva.')

except ImportError:
    print('[INFO] dagster nincs telepítve – pip install dagster')
    print('Logikailag: @asset = Python függvény + Dagster metaadat')

Asset neve: AssetKey(['raw_orders'])


AttributeError: 'AssetsDefinition' object has no attribute 'description'

### Mi történt? – soronkénti magyarázat

```python
@asset                    # (1) Dagster regisztrálja az asset-et
def raw_orders():         # (2) függvénynév → asset kulcs
    return [...]          # (3) visszatérési érték → materializált adat
```

- A `@asset` dekorátor **nem futtatja le** a függvényt – csak regisztrálja.
- A tényleges futtatáshoz `materialize()` kell (lásd 5. fejezet).
- Dagster nyilvántartja: mikor futott, mi volt az output, volt-e hiba.

## 3. Asset függőségek – upstream/downstream

Ha egy `@asset` függvény paraméterének neve egyezik egy másik asset nevével,
Dagster **automatikusan** felismeri a függőséget és felépíti a gráfot.

Lánc: `raw_orders` → `validated_orders` → `order_summary`

In [ ]:
# 3 összefüggő asset: raw_orders → validated_orders → order_summary
try:
    from dagster import asset

    @asset
    def raw_orders():                              # 1. szint – nincs upstream
        """Nyers rendelési adatok."""
        return [
            {'order_id': 1, 'amount':  150.0, 'status': 'completed'},
            {'order_id': 2, 'amount':  320.5, 'status': 'pending'},
            {'order_id': 3, 'amount':  -10.0, 'status': 'cancelled'},
            {'order_id': 4, 'amount': 9999.0, 'status': 'completed'},
        ]

    @asset                                         # 2. szint – upstream: raw_orders
    def validated_orders(raw_orders):              # paraméternév = upstream asset neve
        """Érvényes rendelések: negatív amount és cancelled kiszűrve."""
        return [
            row for row in raw_orders              # raw_orders automatikusan megérkezik
            if row['amount'] > 0 and row['status'] != 'cancelled'
        ]

    @asset                                         # 3. szint – upstream: validated_orders
    def order_summary(validated_orders):           # paraméternév = upstream asset neve
        """Összesítő statisztika az érvényes rendelésekből."""
        total   = sum(r['amount'] for r in validated_orders)
        count   = len(validated_orders)
        average = total / count if count else 0
        return {'count': count, 'total': total, 'average': round(average, 2)}

    print('3 asset definiálva: raw_orders → validated_orders → order_summary')
    print(f'order_summary upstream-jei: {[dep.asset_key.path[-1] for dep in order_summary.asset_deps]}')

except ImportError:
    print('[INFO] dagster nincs telepítve – pip install dagster')

### Hogyan épül fel automatikusan az asset gráf?

Dagster a Python **paraméternevekből** olvassa ki a függőségeket:

```
validated_orders(raw_orders)       → raw_orders a upstream
order_summary(validated_orders)    → validated_orders a upstream
```

Nincs szükség explicit `>>` vagy `set_upstream()` hívásra.
A gráf **mindig szinkronban van a kóddal** – nem lehet elfelejteni frissíteni.

In [ ]:
# Asset gráf szimulálása Python dict-tel (dagster import nélkül)
# Topológiai sorrendezés – BFS/Kahn algoritmus egyszerűsítve

# Függőségi térkép: asset → upstream asset-ek listája
dependencies = {
    'raw_orders':       [],                    # nincs upstream
    'validated_orders': ['raw_orders'],        # raw_orders kell előbb
    'order_summary':    ['validated_orders'],  # validated_orders kell előbb
}

def topological_sort(deps):
    """Kahn-algoritmus: topológiai sorrend meghatározása."""
    in_degree = {node: len(upstream) for node, upstream in deps.items()}
    queue     = [n for n, d in in_degree.items() if d == 0]  # 0 upstream → elsőként fut
    order     = []
    while queue:
        node = queue.pop(0)
        order.append(node)
        # csökkentjük azon node-ok in_degree-jét, amelyek erre támaszkodnak
        for n, upstream in deps.items():
            if node in upstream:
                in_degree[n] -= 1
                if in_degree[n] == 0:
                    queue.append(n)
    return order

execution_order = topological_sort(dependencies)
print('Automatikus futtatási sorrend:')
for i, asset_name in enumerate(execution_order, 1):
    ups = dependencies[asset_name] or ['(nincs)'] 
    print(f'  {i}. {asset_name:<22}  upstream: {ups}')

## 4. Resource fogalma – miért nem importálunk direkt DB connection-t?

**Probléma:** ha az asset közvetlenül importálja a DB kapcsolatot, nem lehet:
- teszteléskor mock-ot használni
- különböző környezetekben (dev/staging/prod) más adatbázist beállítani

**Megoldás:** `ConfigurableResource` – a kapcsolat konfigurációját kívülről adjuk be,
így az asset kódja **nem változik** dev vs prod között, csak a konfiguráció.

In [ ]:
# DuckDBResource mint ConfigurableResource demo
try:
    import duckdb
    from dagster import ConfigurableResource, asset, materialize

    class DuckDBResource(ConfigurableResource):   # Dagster kezeli az életciklust
        database: str = ':memory:'                # konfigurálható paraméter

        def get_connection(self):
            return duckdb.connect(self.database)   # minden híváskor friss kapcsolat

    @asset
    def db_demo(duckdb_resource: DuckDBResource):  # resource típusannotációval injektálva
        con = duckdb_resource.get_connection()        # asset NEM tudja, hol van az adatbázis
        con.execute('CREATE TABLE t AS SELECT 42 AS x')
        result = con.execute('SELECT x FROM t').fetchone()[0]
        return result

    # Futtatás: dev konfigurációval (in-memory)
    dev_result = materialize(
        [db_demo],
        resources={'duckdb_resource': DuckDBResource(database=':memory:')},
    )
    print(f'Dev materializáció sikeres: {dev_result.success}')
    # Prod esetén csak a resources={} sort cseréljük: DuckDBResource(database='/prod/data.db')
    print('Prod esetén: DuckDBResource(database="/prod/data.db") – az asset kódja változatlan!')

except ImportError:
    print('[INFO] dagster / duckdb nincs telepítve – pip install dagster dagster-duckdb duckdb')
    print('ConfigurableResource = konfigurálható, cserélhető kapcsolat-objektum')

### Mi a Resource előnye?

| | Közvetlen import | ConfigurableResource |
|---|---|---|
| Tesztelhetőség | Nehéz – valódi DB kell | Könnyű – mock resource |
| Dev/prod szétválasztás | Kódot kell módosítani | Csak konfiguráció változik |
| Dagit UI felügyelet | Nem látható | Látható, naplózott |

```python
# dev:
resources={'duckdb_resource': DuckDBResource(database=':memory:')}
# prod:
resources={'duckdb_resource': DuckDBResource(database='/data/prod.db')}
# Az asset kódja MINDKÉT esetben azonos!
```

## 5. `materialize()` – hogyan futtatjuk le az asset-eket?

`materialize()` a Dagster "végrehajtó" függvénye notebookban és tesztekben.
Automatikusan meghatározza a futtatási sorrendet a függőségi gráfból,
és összegyűjti az összes eseményt (metadata, logok, hibák).

In [ ]:
# materialize() hívás szimulálása
try:
    from dagster import asset, materialize

    # Egyszerű asset-lánc – dependency a paraméternévből következik
    @asset
    def raw_orders():
        return [{'id': 1, 'amt': 100}, {'id': 2, 'amt': 200}]

    @asset
    def validated_orders(raw_orders):            # upstream: raw_orders
        return [r for r in raw_orders if r['amt'] > 0]

    # materialize: mindkét asset előállítása a helyes sorrendben
    result = materialize([raw_orders, validated_orders])

    print(f'Sikeres materializáció: {result.success}')
    # Az összes asset materializációs esemény listázása
    mat_events = [
        e for e in result.all_node_events
        if e.event_type_value == 'ASSET_MATERIALIZATION'
    ]
    print(f'Materializált asset-ek száma: {len(mat_events)}')
    for e in mat_events:
        print(f'  - {e.asset_key.path[-1]}')

except ImportError:
    print('[INFO] dagster nincs telepítve – pip install dagster')
    print('materialize([raw_orders, validated_orders]) = mindkettőt lefuttatja sorban')

### Mi történt? – materializáció vs task futtatás

| | Task futtatás (Airflow) | Materializáció (Dagster) |
|---|---|---|
| Mit tárol? | Task futási esemény | Asset + futási esemény |
| Újrafuttatás | Mindig újra fut | Csak ha stale (FreshnessPolicy) |
| Lineage | Nincs (csak DAG) | Van – asset szinten |
| UI-ban látható | Run history | Asset catalog + Run history |

A `result.success` után a Dagit Asset Catalog-ban látható az asset utolsó materializációjának ideje.

## 6. Metadata output – megfigyelhetőség

`context.add_output_metadata()` segítségével az asset bármilyen metaadatot
csatolhat a materializációhoz: sorszám, séma, előnézet stb.
Ez jelenik meg a Dagit UI-ban az Asset Catalog részletes nézetén.

In [ ]:
# add_output_metadata demo: rowCount, preview, schema
try:
    from dagster import asset, materialize, MetadataValue

    @asset
    def validated_orders_with_meta(context):     # context: Dagster futási környezet
        rows = [
            {'order_id': 1, 'amount': 150.0, 'status': 'completed'},
            {'order_id': 2, 'amount': 320.5, 'status': 'completed'},
        ]
        # Metadata csatolása a materializációhoz – Dagit UI-ban jelenik meg
        context.add_output_metadata({
            'rowCount': MetadataValue.int(len(rows)),              # sorok száma
            'preview':  MetadataValue.json(rows[:2]),             # első 2 sor előnézete
            'schema':   MetadataValue.json(list(rows[0].keys())), # oszlopnevek
        })
        return rows

    result = materialize([validated_orders_with_meta])
    print(f'Materializáció sikeres: {result.success}')

    # Metadata kinyerése az eseményekből
    for event in result.all_node_events:
        if event.event_type_value == 'ASSET_MATERIALIZATION':
            meta = event.asset_materialization.metadata
            print('\nCsatolt metadata:')
            for k, v in meta.items():
                print(f'  {k}: {v.value}')

except ImportError:
    print('[INFO] dagster nincs telepítve – pip install dagster')
    print('add_output_metadata = rowCount, preview, schema csatolása a materializációhoz')

### Miért hasznos a metadata?

- **Adatminőség monitoring:** `rowCount` hirtelen csökken → riasztás lehetséges
- **Lineage dokumentáció:** a séma változásai automatikusan naplózódnak
- **Debugolás:** `preview` megmutatja az első sorokat UI-ból anélkül, hogy le kéne futtatni

Dagit UI → Asset Catalog → asset kiválasztása → **Materialization** fül → metadata táblázat

## 7. `FreshnessPolicy` – mikor "elavult" egy asset?

A `FreshnessPolicy` meghatározza, hogy egy asset maximálisan mennyi ideig lehet
"friss" materializáció nélkül. Ha lejár, a Dagit `stale` jelzéssel mutatja.

Ez az adatmérnöki **SLA** Dagster-szintű megvalósítása.

In [ ]:
# FreshnessPolicy demo: maximum_lag_minutes, cron_schedule
try:
    from dagster import asset, FreshnessPolicy

    @asset(
        freshness_policy=FreshnessPolicy(
            maximum_lag_minutes=60,          # max 1 óra lehet "régi"
            cron_schedule='0 * * * *',       # óránkénti frissítést vár el
        )
    )
    def hourly_sales_report():
        """Óránkénti értékesítési riport – 1 óránál régebben stale."""
        return {'sales': 42000, 'orders': 17}

    @asset(
        freshness_policy=FreshnessPolicy(
            maximum_lag_minutes=60 * 24,     # max 24 óra lehet "régi"
        )
    )
    def daily_kpi_snapshot():
        """Napi KPI pillanatkép – 24 óránál régebben stale."""
        return {'revenue': 1_200_000, 'churn_rate': 0.03}

    print(f'hourly_sales_report freshness: {hourly_sales_report.freshness_policy}')
    print(f'daily_kpi_snapshot  freshness: {daily_kpi_snapshot.freshness_policy}')
    print('Ha az asset nem materializálódott a megadott időn belül → Dagit: STALE')

except ImportError:
    print('[INFO] dagster nincs telepítve – pip install dagster')
    print('FreshnessPolicy(maximum_lag_minutes=60) = 1 óra után stale')

### Mi történt? – SLA vs FreshnessPolicy

| | Hagyományos SLA | Dagster FreshnessPolicy |
|---|---|---|
| Hol van definiálva? | Dokumentáció / monitoring tool | A kódban, az asset mellett |
| Verziókövetés? | Nehézkes | Automatikus (git) |
| Láthatóság | Külön rendszer | Dagit Asset Catalog |
| Riasztás | Manuális setup | Beépített Dagit jelzés |

A `cron_schedule` opcionális: megmondja Dagsternek, mikor **kellene** a frissítés,
és ez alapján számolja ki, mikor lesz az asset `stale`.

## 8. Sensor – event-driven materializáció

A **sensor** figyeli a külső eseményeket és ezek alapján indít materializációt.
Nem időalapú (cron), hanem **eseményalapú** – például: új fájl érkezett, DB változott.

A `@asset_sensor` egy adott asset materializációját figyeli,
és ha az bekövetkezik, elindít egy másik asset-et.

In [ ]:
# @asset_sensor demo szimulálva (try/except) + RunRequest
try:
    from dagster import (
        asset, asset_sensor, AssetKey,
        SensorEvaluationContext, RunRequest, SkipReason
    )

    @asset
    def raw_orders():
        return [{'id': 1, 'amount': 100}]

    @asset
    def validated_orders(raw_orders):
        return [r for r in raw_orders if r['amount'] > 0]

    # Sensor: figyeli a raw_orders asset-et
    # Ha raw_orders materializálódott → elindítja a validated_orders előállítását
    @asset_sensor(
        asset_key=AssetKey('raw_orders'),         # ezt az asset-et figyeli
        job_name='__ASSET_JOB',                   # Dagster belső job neve
    )
    def raw_orders_sensor(context: SensorEvaluationContext, asset_event):
        """Ha raw_orders frissült, elindítja a validated_orders-t."""
        latest_run_id = context.cursor                     # utolsó feldolgozott futás
        if asset_event.run_id != latest_run_id:            # új materializáció?
            yield RunRequest(run_key=asset_event.run_id)   # igen → futtatás kérés
        else:
            yield SkipReason('Nincs új materializáció.')   # nem → kihagyás

    print('asset_sensor definiálva: raw_orders változása → validated_orders újrafuttatása')
    print(f'Sensor neve: {raw_orders_sensor.name}')

except ImportError:
    print('[INFO] dagster nincs telepítve – pip install dagster')
    print('@asset_sensor: figyeli az upstream asset materializációját, RunRequest-et küld')

### Sensor vs Schedule: mikor melyiket válasszuk?

| | Schedule (cron) | Sensor (event-driven) |
|---|---|---|
| Trigger | Idő alapján | Esemény alapján |
| Példa | Minden nap éjfélkor | Ha új CSV érkezett az S3-ba |
| Késés | Rögzített (pl. reggel 6) | Minimális – azonnal fut |
| Felesleges futás | Igen, ha nincs adat | Nem – csak ha van esemény |

**Ökölszabály:**
- Adatforrás **rendszeres ütemben** frissül → `Schedule`
- Adatforrás **esemény hatására** frissül (fájl, webhook, DB trigger) → `Sensor`

## 9. `Definitions` – a projekt összefoglaló konfigurációja

A `Definitions` objektum fogja össze az összes asset-et, resource-t és sensor-t.
Ez az egyetlen belépési pont, amit a `dagster dev` parancs betölt.

```bash
dagster dev -f pipeline.py   # → http://localhost:3000
```

In [ ]:
# Definitions(assets=[...], resources={...}, sensors=[...]) demo
try:
    from dagster import (
        asset, asset_sensor, AssetKey,
        SensorEvaluationContext, RunRequest, SkipReason,
        ConfigurableResource, Definitions, materialize
    )
    import duckdb

    # Resource definíció
    class DuckDBResource(ConfigurableResource):
        database: str = ':memory:'
        def get_connection(self):
            return duckdb.connect(self.database)

    # Asset-ek
    @asset
    def raw_orders():
        return [{'id': 1, 'amount': 100}, {'id': 2, 'amount': 200}]

    @asset
    def validated_orders(raw_orders):
        return [r for r in raw_orders if r['amount'] > 0]

    @asset
    def order_summary(validated_orders):
        return {'count': len(validated_orders), 'total': sum(r['amount'] for r in validated_orders)}

    # Sensor
    @asset_sensor(asset_key=AssetKey('raw_orders'), job_name='__ASSET_JOB')
    def raw_orders_sensor(context: SensorEvaluationContext, asset_event):
        if asset_event.run_id != context.cursor:
            yield RunRequest(run_key=asset_event.run_id)
        else:
            yield SkipReason('Nincs új materializáció.')

    # Definitions – minden egyben: ez töltődik be a dagster dev paranccsal
    defs = Definitions(
        assets=[raw_orders, validated_orders, order_summary],  # összes asset
        resources={'duckdb_resource': DuckDBResource()},        # resource-ok
        sensors=[raw_orders_sensor],                           # sensorok
    )

    print('Definitions objektum létrehozva:')
    print(f'  Asset-ek:   {[a.key.path[-1] for a in defs.get_all_asset_defs()]}')
    print(f'  Resource-ok: {list(defs.resources.keys())}')
    print(f'  Sensor-ok:  {[s.name for s in defs.sensors]}')

except ImportError:
    print('[INFO] dagster nincs telepítve – pip install dagster duckdb')
    print('Definitions = az összes asset + resource + sensor egy belépési pontban')

## 10. Összefoglalás – mikor válasszuk a Dagster-t?

### Döntési fa

```
Van adatobjektum (tábla, fájl, modell), amit nyilvántartani és figyelni kell?
├── IGEN → Dagster (asset-centric)
│     Fontos a lineage, freshness, metadata, sensor?
│     ├── IGEN → Dagster biztosan jó választás
│     └── NEM  → Prefect is elég lehet
└── NEM  → Csak végrehajtási sorrend kell?
      ├── Egyszerű, időalapú → Airflow
      └── Dinamikus, eseményalapú → Prefect
```

### Három eszköz összehasonlítása

| | Airflow | Prefect | Dagster |
|---|---|---|---|
| Paradigma | Task-centric | Task-centric | Asset-centric |
| Lineage | Korlátozott | Korlátozott | Beépített |
| FreshnessPolicy | Nincs | Nincs | Van |
| Sensor | Van (polling) | Van | Van (asset-aware) |
| Tesztelhetőség | Nehéz | Közepes | Kiváló (Resource) |
| Tanulási görbe | Meredek | Lapos | Közepes |

**Dagster erőssége:** az adatobjektumok és a kód **egy helyen** vannak definiálva,
a lineage, freshness és metadata **automatikusan** következnek a kód struktúrájából.